# 03 · Filter & Rank — run the shared multi-layer filter (binder cutoffs)

**Standard slot:** *filter & rank* via `shared/filtering_pipeline.py` — the same module all 25
projects use. **For Project 09** you build `fp.Design` **binder** objects from each modality pool, call
`fp.run_pipeline(..., design_type="binder")`, and `fp.report(...)` the survival funnel + ranked CSV,
**per modality** (linear / macrocycle / mini-protein foil) so the comparisons are fair (D3 part 1). The
**Boltz-2 affinity score rides along for ranking only — it is never a pass/fail and never a K_D.**

> **Do not fork the module into this project.** Iterate against `shared/filtering_pipeline.py` and PR
> improvements back. This notebook *imports* it.

Run `00`–`02` first so `results/linear_designs.csv`, `results/macrocycle_designs.csv`, and
`results/miniprotein_foil_designs.csv` exist.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## Load the shared filtering pipeline
This is the cohort's shared module — improvements here are pull-requested back for everyone. Note the
`"binder"` cutoffs: scRMSD ≤ 2.5, pLDDT ≥ 80, pae ≤ 10, rosetta_dG ≤ −30, sc ≥ 0.6. (The Boltz-2
affinity score is **not** a cutoff — it is a relative rank we keep alongside for prioritization.)

In [ ]:
import filtering_pipeline as fp
import pandas as pd

print("Loaded shared filtering_pipeline from:", fp.__file__)
print("binder cutoffs:", fp.DEFAULT_CUTOFFS["binder"])

## Build `Design` (binder) objects from the modality pools

Map each pool row onto `fp.Design` with `design_type="binder"`. The metrics drive the layers:
`scrmsd`/`plddt`/`pae_interaction` (Layer 1 self-consistency), and `rosetta_dG`/`shape_complementarity`/`solubility`
(Layer 3 physics). We keep `modality`, `cyclic`, `cleft_overlap`, and the Boltz-2 rank in `extra` for
the modality comparison + cleft analysis in notebook 04. (Mock has no independent orthogonal predictor,
so we run Layers 1+3 here; on Colab add a second predictor — e.g. AF2 ↔ Boltz-2 — for Layer 2.)

In [ ]:
import os
import pandas as pd

# Regenerate the pools if a fresh session lost them (deterministic mock).
NEEDED = ["results/linear_designs.csv", "results/macrocycle_designs.csv",
          "results/miniprotein_foil_designs.csv"]
if not all(os.path.exists(p) for p in NEEDED):
    import peptide_tools as pt
    TARGET, CLEFT = "MDM2", pt.parse_cleft("A54,A67,A73,A93,A100")
    def _mk(designs, p):
        rows = []
        for d in designs:
            rdg = -45.0 + (pt._hashints("dG", d.design_id) % 40)
            rows.append(dict(design_id=d.design_id, modality=d.modality, cyclic=d.cyclic,
                             length=d.length, sequence=d.sequence, plddt=d.plddt,
                             pae_interaction=d.pae_interaction, scrmsd=d.scrmsd,
                             shape_complementarity=d.shape_complementarity,
                             boltz_affinity_score=d.boltz_affinity_score,
                             rosetta_dG=round(float(rdg), 2), solubility=0.3,
                             cleft_overlap=pt.cleft_overlap(d.contact_residues, d.cleft),
                             synthetic=d.synthetic))
        pd.DataFrame(rows).to_csv(p, index=False)
    lin = []
    for L in [8, 12, 16, 20]:
        lin += pt.design_peptide(TARGET, CLEFT, length=L, cyclic=False, n=15, tool="mock")
    cyc = []
    for L in [8, 11, 14]:
        cyc += pt.design_peptide(TARGET, CLEFT, length=L, cyclic=True, n=15, tool="mock")
    foil = pt.design_miniprotein_foil(TARGET, CLEFT, n=30, tool="mock")
    for g in (lin, cyc, foil):
        pt.score_designs(g, tool="mock")
    _mk(lin, NEEDED[0]); _mk(cyc, NEEDED[1]); _mk(foil, NEEDED[2])

df_lin  = pd.read_csv(NEEDED[0])
df_cyc  = pd.read_csv(NEEDED[1])
df_foil = pd.read_csv(NEEDED[2])

def row_to_binder(r):
    return fp.Design(
        design_id=str(r["design_id"]), sequence=str(r.get("sequence", "")), design_type="binder",
        plddt=r.get("plddt"), pae_interaction=r.get("pae_interaction"), scrmsd=r.get("scrmsd"),
        scrmsd_orthogonal=r.get("scrmsd"),   # mock: reuse scrmsd as a stand-in; use a 2nd predictor on Colab
        rosetta_dG=r.get("rosetta_dG"), shape_complementarity=r.get("shape_complementarity"),
        solubility=r.get("solubility", 0.3),
        extra={"modality": r.get("modality"), "cyclic": bool(r.get("cyclic")),
               "cleft_overlap": r.get("cleft_overlap"),
               "boltz_affinity_score": r.get("boltz_affinity_score")},
    )

binders_lin  = [row_to_binder(r) for _, r in df_lin.iterrows()]
binders_cyc  = [row_to_binder(r) for _, r in df_cyc.iterrows()]
binders_foil = [row_to_binder(r) for _, r in df_foil.iterrows()]
print(f"built {len(binders_lin)} linear + {len(binders_cyc)} macrocycle + {len(binders_foil)} mini-protein Designs")

## Run the pipeline — per modality (fair comparison)

`run_pipeline(design_type="binder")` applies the binder cutoffs in order and returns a ranked
DataFrame with survival counts in `df.attrs`. We run **each modality separately** so the
survival-at-each-layer funnels are comparable. We use Layers 1+3 here (mock has no independent
orthogonal source; add Layer 2 on Colab with a second predictor).

In [ ]:
def run_one(designs, label):
    df = fp.run_pipeline(designs, design_type="binder", use_layers=(1, 3))
    df["modality"] = label
    surv = df.attrs["survival"]; n = df.attrs["n_total"]
    passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: {n} designs, survival {surv}, all-layers hit rate = {passed}/{n} ({100*passed/max(n,1):.1f}%)")
    return df

ranked_lin  = run_one(binders_lin,  "linear")
ranked_cyc  = run_one(binders_cyc,  "macrocycle")
ranked_foil = run_one(binders_foil, "miniprotein")

ranked = pd.concat([ranked_lin, ranked_cyc, ranked_foil], ignore_index=True).sort_values(
    ["layers_passed", "score"], ascending=False).reset_index(drop=True)
ranked.to_csv("results/all_ranked.csv", index=False)
print("\nwrote results/all_ranked.csv", ranked.shape)
ranked.head(10)[["design_id", "modality", "layers_passed", "score",
                 "scrmsd", "plddt", "pae_interaction", "rosetta_dG"]]

## Survival-at-each-layer via `report()`

`report()` prints the hit-rate accounting and draws the survival funnel. Here we report the **combined**
pool for one comparable figure; the per-modality runs above are the rigorous version. Read the bars as
a funnel: steep drops show which layer discriminates.

In [ ]:
import matplotlib
matplotlib.use("Agg")  # headless-safe; Colab still displays inline

all_binders = binders_lin + binders_cyc + binders_foil
df_all = fp.run_pipeline(all_binders, design_type="binder", use_layers=(1, 3))
top = fp.report(df_all, top_n=15, save_prefix="results/p09")
print("\nsaved results/p09_survival.png + results/p09_ranked.csv")
top

## Honest hit-rate accounting (per modality)

Report `N passing all layers / N generated` for **each** modality — this is the number the comparisons
in notebook 04 build on. Survival is *enrichment*, not *correctness*; and for short peptides predicted
affinity is unreliable (the Boltz-2 score is a rank, not a K_D). Mock numbers are SYNTHETIC.

In [ ]:
for label, df in [("linear", ranked_lin), ("macrocycle", ranked_cyc), ("miniprotein", ranked_foil)]:
    n = len(df); passed = int((df["layers_passed"] >= 3).sum())
    print(f"{label:12s}: layers_passed distribution {df['layers_passed'].value_counts().sort_index().to_dict()}")
    print(f"{'':12s}  all-layers survivors = {passed}/{n} ({100*passed/max(n,1):.1f}%)  [SYNTHETIC if mock]")

## D3 (part 1) checklist
- [ ] `results/all_ranked.csv` produced by the **shared** module (`design_type="binder"`), not a one-off script.
- [ ] Survival-at-each-layer reported **per modality** (funnel figure `results/p09_survival.png`).
- [ ] Honest hit-rate accounting (N pass / N generated) for linear, macrocycle, and mini-protein foil.
- [ ] Boltz-2 affinity kept as a **rank only** (never a cutoff, never a K_D); mapping assumptions written down.

**Next:** `04_validate.ipynb` — the linear-vs-cyclic + peptide-vs-protein comparisons.